# Can a cheap frequency detector find *where* an attack hit? — a measured answer

[`../06-Attack-Benchmark/`](../06-Attack-Benchmark/) predicted that the tile gate in
[`../05-Noise-Gate/`](../05-Noise-Gate/) — which scores high-frequency energy —
would catch L∞ attacks but be blind to a **low-frequency / Nightshade-Glaze** poison.
This notebook **tests that prediction**, and tests the obvious fix (widen the single HF band
into a multi-band bank).

**Headline result, measured below — the obvious fix does not work.**

| detector | k | HF-attack IoU | LF-attack IoU | clean FP |
|---|---|---|---|---|
| HF only (the gate) | 3 | 0.32 | **0.05** | 16% |
| multi-band | 3 | 0.36 | **0.24** | **47%** |
| multi-band | 6 | 0.06 | **0.06** | 12% |

The gate's blind spot is **real** (LF 0.05 vs HF 0.32) — the benchmark's prediction holds.
Multi-band *looks* like it rescues the low-frequency case (0.05 → 0.24), but only by flagging
**47% of clean tiles**. Pull it back to a comparable false-positive rate and the advantage
vanishes (**0.06 vs 0.05**). The gain was bought with false positives, not detection.

**Any comparison at a fixed threshold is meaningless — you must compare at matched FP.** That is
the money figure in §4.

This is a **negative result**, and it is the point: it turns "use a learned per-tile classifier"
(already in the 05-Noise-Gate research log) from a hunch into an evidence-backed next step.

### What *is* solid here

Three engineering fixes, independently verified, that hold whatever the scoring statistic:

1. **No tiling.** Blurring is translation-invariant, so blur the *whole* image once and
   `avg_pool` the residual. Tiling first zero-pads every tile, fabricating a black neighbour:
   on interior tiles that inflates the σ=8 band by **24×**. A real bug in the first version.
2. **Separable convolution.** The Gaussian is an outer product, so a 49×49 2D conv becomes two
   1D convs (~25× fewer FLOPs). Exact to ~1e-6.
3. **Native batching.** `[N,3,H,W]` in one pass — no `unfold`, no ThreadPoolExecutor, no
   multi-GPU chunking. Timed live in §2.

## 0. Setup

In [ ]:
!pip install -q diffusers transformers accelerate

In [ ]:
import torch, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
from torchvision import transforms
from PIL import Image, ImageDraw, ImageFilter
import urllib.request, os, glob, time

DEV = "cuda" if torch.cuda.is_available() else "cpu"
SIZE = 512
torch.manual_seed(0)
print("device:", DEV)

## 1. The detector, tile-free

`band_maps` blurs the **whole image** and returns the energy of each octave band at **full
resolution** — no tiles, so no fabricated tile-seam edges. Everything downstream is a *choice of
how to pool that map*: a 4×4 grid, a 32×32 grid, or a sliding window for arbitrary shapes. The
expensive part is computed once.

In [ ]:
SIGMAS  = [1.0, 2.0, 4.0, 8.0, 16.0]      # octave bank, fine -> coarse
N_BANDS = len(SIGMAS)

def gauss1d(sigma):
    ksize = int(6*sigma) | 1                                  # odd kernel wide enough for sigma
    ax = torch.arange(ksize) - ksize // 2
    g = torch.exp(-(ax.float()**2) / (2*sigma**2))
    return g / g.sum()

def blur_sep(x, sigma):
    """Separable Gaussian: two 1D convs instead of one ksize^2 2D conv (~25x fewer FLOPs)."""
    g = gauss1d(sigma).to(x.device, x.dtype)
    k = g.numel(); p = k // 2; C = x.shape[1]
    x = F.conv2d(x, g.view(1,1,1,k).repeat(C,1,1,1), padding=(0,p), groups=C)
    x = F.conv2d(x, g.view(1,1,k,1).repeat(C,1,1,1), padding=(p,0), groups=C)
    return x

def band_maps(x):
    """[N,3,H,W] -> [N,B,H,W]: |energy| per octave band, FULL resolution.
    Blur is computed on the whole image, so interior pixels see their REAL neighbours."""
    bl = [blur_sep(x, s) for s in SIGMAS]
    bands = [x - bl[0]] + [bl[i] - bl[i+1] for i in range(len(bl)-1)]
    return torch.stack([b.abs().mean(1) for b in bands], 1)     # mean over colour

def grid_pool(bm, grid=4):
    """Pool the band maps onto a grid x grid lattice -> [N, grid*grid, B].
    Same statistic the tiled gate computed, minus the padding lie. The grid is now a free
    parameter: 4x4 or 32x32 costs the same, because band_maps was already computed."""
    p = F.avg_pool2d(bm, kernel_size=bm.shape[-1] // grid)      # [N,B,grid,grid]
    return p.flatten(2).transpose(1, 2)                         # [N, grid*grid, B]

print(f"{N_BANDS}-band bank, sigmas {SIGMAS}")

## 2. Sanity checks: separability is exact, and batching is much faster

Verified live rather than asserted.

In [ ]:
# --- separable == dense 2D, to float precision ---
def gauss2d(sigma):
    g = gauss1d(sigma); k = g.numel()
    return torch.outer(g, g).view(1,1,k,k).repeat(3,1,1,1)

x = torch.rand(1,3,SIZE,SIZE, device=DEV)
for s in SIGMAS:
    k2 = gauss2d(s).to(DEV)
    dense = F.conv2d(x, k2, padding=k2.shape[-1]//2, groups=3)
    print(f"sigma={s:<5} separable vs dense 2D: max abs diff = {(dense - blur_sep(x,s)).abs().max():.2e}")

# --- old tiled path (loop images, dense conv per tile) vs new (one batched pass) ---
def tile_image(x, grid=4):
    t = x.shape[-1] // grid
    p = x.unfold(2,t,t).unfold(3,t,t)
    return p.permute(0,2,3,1,4,5).reshape(-1,3,t,t).contiguous()

def old_tiled(batch):
    out = []
    for i in range(len(batch)):                      # Python loop over images
        tiles = tile_image(batch[i:i+1])             # + a full-image copy
        bl = [F.conv2d(tiles, gauss2d(s).to(DEV), padding=gauss2d(s).shape[-1]//2, groups=3)
              for s in SIGMAS]
        bands = [tiles - bl[0]] + [bl[j]-bl[j+1] for j in range(len(bl)-1)]
        out.append(torch.stack([b.abs().mean(dim=(1,2,3)) for b in bands], 1))
    return torch.cat(out)

batch = torch.rand(12,3,SIZE,SIZE, device=DEV)
for fn, name in [(old_tiled,"old: tiled + loop"), (lambda b: grid_pool(band_maps(b)), "new: batched pool")]:
    with torch.no_grad():
        fn(batch)
        if DEV=="cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(3): fn(batch)
        if DEV=="cuda": torch.cuda.synchronize()
        print(f"{name:20s}: {(time.perf_counter()-t0)/3*1000:8.1f} ms for 12 images")

### The padding bug, quantified

`conv2d` zero-pads **each tile independently**, so an interior tile is told its neighbours are
black. At σ=8 the kernel is 49 px wide — 24 px of fabricated border on a 128 px tile. The coarse
bands end up measuring that fake edge instead of the image.

In [ ]:
# a smooth image: the regime the low-frequency bands are supposed to measure
yy, xx = torch.meshgrid(torch.linspace(0,1,SIZE), torch.linspace(0,1,SIZE), indexing='ij')
smooth = (0.5 + 0.3*torch.sin(6*xx)*torch.cos(5*yy)).expand(3,-1,-1).unsqueeze(0).to(DEV)

with torch.no_grad():
    E_tiled  = old_tiled(smooth)                     # per-tile, WITH per-tile padding
    E_pooled = grid_pool(band_maps(smooth))[0]       # per-tile, blurred on the whole image

interior = [5,6,9,10]                                # tiles whose neighbours actually exist
print("inflation of the tiled score on INTERIOR tiles (tiled / true):")
for b in range(N_BANDS):
    print(f"  band {b} (sigma={SIGMAS[b]:>4}): {(E_tiled[interior,b]/E_pooled[interior,b]).mean():5.2f}x")

## 3. Images and a spectrally-varied attack zoo

Every attack is normalised to the **same RMS inside the mask**, so the comparison across spectra
is fair — otherwise the broadband attack simply carries more energy and "wins" for free (a
mistake that cost me a wrong conclusion earlier: my first LF attack was 4× weaker in RMS than
the HF one).

The masks are deliberately **not rectangles**: an irregular blob, a ring (with a hole), and a
thin scribble.

In [ ]:
IMAGE_SOURCE = "web"                 # "web" | "folder"
IMAGE_DIR    = "/kaggle/input"
RMS          = 0.05                  # perturbation energy, equal for every attack

_WEB = ["https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"] + [
  f"https://raw.githubusercontent.com/EliSchwartz/imagenet-sample-images/master/{n}"
  for n in ["n07747607_orange.JPEG","n02690373_airliner.JPEG","n03095699_container_ship.JPEG",
            "n04285008_sports_car.JPEG","n07753592_banana.JPEG"]]

tt = transforms.Compose([transforms.Resize((SIZE,SIZE)), transforms.ToTensor()])
def load(p):
    if str(p).startswith("http"):
        fn = "/tmp/"+os.path.basename(p)
        if not os.path.exists(fn): urllib.request.urlretrieve(p, fn)
        p = fn
    return tt(Image.open(p).convert("RGB")).unsqueeze(0).to(DEV)

if IMAGE_SOURCE == "folder":
    paths = sorted(f for f in glob.glob(os.path.join(IMAGE_DIR,"**","*"), recursive=True)
                   if f.lower().endswith((".jpg",".jpeg",".png",".bmp")))[:6]
    raw = [load(p) for p in paths]
else:
    raw = [load(u) for u in _WEB]
print(f"{len(raw)} images loaded")

def shape_mask(kind, seed=0):
    """Arbitrary shapes -- none of them tile-aligned, none of them rectangles."""
    rng = np.random.default_rng(seed)
    im = Image.new("L", (SIZE,SIZE), 0); d = ImageDraw.Draw(im)
    if kind == "blob":
        cx, cy = rng.uniform(0.3,0.7,2)*SIZE
        a = np.sort(rng.uniform(0,2*np.pi,9)); r = rng.uniform(0.12,0.30,9)*SIZE
        d.polygon([(float(cx+ri*np.cos(ai)), float(cy+ri*np.sin(ai))) for ai,ri in zip(a,r)], fill=1)
    elif kind == "ring":
        d.ellipse([120,120,392,392], fill=1); d.ellipse([190,190,322,322], fill=0)
    elif kind == "scribble":
        pts = [(80,300)] + [(80+i*60, 300+int(90*np.sin(i*1.3))) for i in range(1,7)]
        d.line(pts, fill=1, width=26, joint="curve")
    return torch.tensor(np.array(im), dtype=torch.float32).view(1,1,SIZE,SIZE).to(DEV)

def attack(x, mask, kind, rms=RMS):
    """hf = broadband (FGSM/PGD-like) | lf = low-frequency (Nightshade/Glaze-like) | mid.
    All normalised to equal RMS inside the mask. The detector never learns which was used."""
    if kind == "hf":  p = torch.randn_like(x).sign()
    elif kind == "lf": p = blur_sep(torch.randn_like(x), 8.0)
    else:              n = torch.randn_like(x); p = blur_sep(n,2.0) - blur_sep(n,4.0)
    mm = mask.expand_as(p) > 0.5
    p = p / ((p[mm]**2).mean().sqrt() + 1e-8)
    return (x + rms*p*mask).clamp(0,1)

def true_tiles(mask, grid=4, cover=0.05):
    return (F.avg_pool2d(mask, SIZE//grid).flatten() > cover)

fig, ax = plt.subplots(1,3, figsize=(11,3.6))
for a,s in zip(ax, ["blob","ring","scribble"]):
    a.imshow(shape_mask(s, seed=1)[0,0].cpu(), cmap="gray"); a.set_title(s); a.axis("off")
plt.suptitle("attack regions: arbitrary shapes, not rectangles"); plt.tight_layout(); plt.show()

## 4. The money figure: IoU vs. false positives

A single threshold proves nothing — a detector can always raise IoU by flagging more. So we
**sweep `k`** and plot detection IoU against the clean-image false-positive rate. A genuinely
better detector sits **up and to the left**.

Both detectors are calibrated the same way the gate does it: robust `median + k·MAD` per band
over trusted-clean images (0–3), then scanned over held-out images (4–5).

In [ ]:
CLEAN, TEST = raw[:4], raw[4:]

def calibrate(clean, bands):
    E = torch.cat([grid_pool(band_maps(x))[0] for x in clean])[:, bands]   # [Nc*16, |bands|]
    med = E.median(0).values
    mad = (E - med).abs().median(0).values + 1e-8
    return med, 1.4826*mad

def flags(x, cal, bands, k):
    E = grid_pool(band_maps(x))[0][:, bands]
    return ((E - cal[0]) / cal[1]).amax(1) > k        # flag if ANY band is anomalous

def iou(a, b):
    u = (a|b).sum().item()
    return (a&b).sum().item()/u if u else 1.0

DETECTORS = {"HF only (the gate)": [0], "multi-band": [0,1,2,3,4]}
KS = [1.0,1.5,2.0,2.5,3.0,4.0,5.0,6.0,8.0,10.0]

with torch.no_grad():
    curves = {}
    for name, bands in DETECTORS.items():
        cal = calibrate(CLEAN, bands)
        for kind in ["hf","lf"]:
            pts = []
            for k in KS:
                ious, fps = [], []
                for x in TEST:
                    for sd in [1,2,3]:
                        m = shape_mask("blob", seed=sd)
                        ious.append(iou(flags(attack(x,m,kind), cal, bands, k), true_tiles(m)))
                    fps.append(flags(x, cal, bands, k).float().mean().item())
                pts.append((np.mean(fps)*100, np.mean(ious)))
            curves[(name,kind)] = pts

fig, axes = plt.subplots(1,2, figsize=(12,4.6), sharey=True)
for ax, kind, title in zip(axes, ["hf","lf"],
                           ["HF attack (FGSM/PGD-like)", "LF attack (Nightshade-like) — the hard one"]):
    for name in DETECTORS:
        p = np.array(curves[(name,kind)])
        ax.plot(p[:,0], p[:,1], "o-", label=name)
    ax.set_xlabel("clean-image false positives (% of tiles)"); ax.set_title(title)
    ax.grid(alpha=.3); ax.legend()
axes[0].set_ylabel("detection IoU")
plt.suptitle("Better = up and to the LEFT. Compare at matched FP, never at fixed k.")
plt.tight_layout(); plt.show()

print("On the LF attack the two curves lie on top of each other: at any given false-positive")
print("budget, the multi-band bank buys you no extra low-frequency detection.")

## 5. Arbitrary shapes: the machinery works, the statistic doesn't

Dropping tiles means the anomaly score exists **per pixel**, so nothing forces a rectangle. Use a
**sliding window** (a "tile centred on every pixel") instead of a disjoint grid, z-score against
the image's *own* pixels — cross-image calibration is hopeless here, since median band energy
varies ~6× between clean photos — then clean up with morphological open/close.

This recovers any shape *in principle*. **Measured honestly it does not work**: with the
threshold picked on one dev image and evaluated on held-out images, IoU lands at **0.05–0.16**
with **~15% false positives**. Shown here so the failure is reproducible, not hidden.

In [ ]:
def self_z(x, agg=6.0):
    """Per-pixel anomaly z, calibrated on THIS image's own pixels (the attack is a minority).
    Log space: local energy is multiplicative, so a ratio-shaped attack becomes an additive
    shift independent of how bright or textured the region is."""
    le = torch.log(blur_sep(band_maps(x), agg) + 1e-6)          # sliding-window local mean
    f = le.permute(1,0,2,3).reshape(N_BANDS,-1)
    med = f.median(1).values.view(1,-1,1,1)
    mad = (f - med.view(-1,1)).abs().median(1).values.view(1,-1,1,1) + 1e-8
    return ((le - med)/(1.4826*mad)).amax(1)[0]                 # [H,W]

def _dil(m,r): return F.max_pool2d(m, 2*r+1, stride=1, padding=r)
def _ero(m,r): return -F.max_pool2d(-m, 2*r+1, stride=1, padding=r)

def locate(x, k=1.25, open_r=5, close_r=10):
    """-> bool [H,W]. Any shape: blob, ring, scribble. No grid anywhere."""
    m = (self_z(x) > k).float()[None,None]
    m = _dil(_ero(m, open_r), open_r)        # open  : drop speckle
    m = _ero(_dil(m, close_r), close_r)      # close : fill pinholes
    return m[0,0] > 0.5

with torch.no_grad():
    fig, ax = plt.subplots(3,4, figsize=(14,10))
    for r, shape in enumerate(["blob","ring","scribble"]):
        m = shape_mask(shape, seed=1)
        x = attack(raw[4], m, "hf")
        z, pred = self_z(x), locate(x)
        ax[r,0].imshow(x[0].permute(1,2,0).cpu()); ax[r,0].set_title(f"{shape}: attacked")
        ax[r,1].imshow(m[0,0].cpu(), cmap="gray"); ax[r,1].set_title("truth")
        ax[r,2].imshow(z.cpu(), cmap="hot"); ax[r,2].set_title("per-pixel anomaly z")
        ax[r,3].imshow(pred.cpu(), cmap="gray")
        ax[r,3].set_title(f"located (IoU {iou(pred, m[0,0]>0.5):.2f})")
        for a in ax[r]: a.axis("off")
    plt.suptitle("Shape recovery is unconstrained — but the frequency statistic is too weak to use")
    plt.tight_layout(); plt.show()

    print("held-out clean false-positive rate:")
    for i,x in enumerate(raw):
        print(f"  image {i}: {locate(x).float().mean().item()*100:5.1f}% of pixels flagged")

## 6. Repair, for when a mask *is* trustworthy

The repair half is independent of how the mask was found: given a region, Stable Diffusion
inpaints only that region and copies the rest through. Demonstrated on the **ground-truth mask**,
because §4–5 showed the detector's mask is not yet good enough to drive it — wiring a 15%-FP mask
into an inpainter would repaint healthy image. Guarded so it runs without the weights.

In [ ]:
m = shape_mask("blob", seed=1)
x = attack(raw[4], m, "hf")
att_img  = Image.fromarray((x[0].permute(1,2,0).cpu().numpy()*255).astype(np.uint8))
mask_img = Image.fromarray((m[0,0].cpu().numpy()*255).astype(np.uint8)).filter(ImageFilter.MaxFilter(9))

repaired = None
try:
    from diffusers import StableDiffusionInpaintPipeline
    pipe = StableDiffusionInpaintPipeline.from_pretrained(
        "runwayml/stable-diffusion-inpainting", torch_dtype=torch.float16).to(DEV)
    pipe.set_progress_bar_config(disable=True)
    repaired = pipe(prompt="a natural, photorealistic photo, high quality",
                    image=att_img, mask_image=mask_img,
                    num_inference_steps=30, guidance_scale=7.5).images[0]
except Exception as e:
    print("SD inpaint unavailable:", type(e).__name__, "- showing the mask only")

n = 3 if repaired is None else 4
fig, ax = plt.subplots(1,n, figsize=(4*n,4))
ax[0].imshow(att_img); ax[0].set_title("attacked")
ax[1].imshow(mask_img, cmap="gray"); ax[1].set_title("repair mask (ground truth)")
ax[2].imshow(raw[4][0].permute(1,2,0).cpu()); ax[2].set_title("original (reference)")
if repaired is not None: ax[3].imshow(repaired); ax[3].set_title("SD-repaired")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## Takeaway

1. **The benchmark's prediction is confirmed.** The HF gate localizes a broadband attack
   (IoU 0.32) and is blind to an equal-energy low-frequency one (**0.05**).
2. **The obvious fix fails.** A multi-band bank appears to fix it (0.05 → 0.24) but only by
   flagging 47% of clean tiles; at matched false positives it is no better (0.06 vs 0.05). The
   §4 curves overlap on the LF attack.
3. **Why it fails, and why it is structural.** Natural images are non-stationary: median band
   energy varies ~6× *between* clean photos (0.0048 → 0.0281) and more *within* one photo between
   sky and foliage. A bounded perturbation lifts local energy ~2.5×. **The confound is larger
   than the signal**, so no threshold on hand-designed band energy separates them.
4. **Therefore: learn the statistic.** This is the evidence for the small learned per-tile
   classifier in the [`../05-Noise-Gate/`](../05-Noise-Gate/) research log. A promising untested
   route is a **VAE reconstruction residual** — encode→decode projects onto SD's learned
   natural-image manifold, and off-manifold perturbations should not survive the round trip,
   giving a learned prior instead of a hand-picked band.

**Methodological notes, learned the hard way in this folder:**
- Compare at **matched false-positive rate**. A fixed threshold flatters whichever detector flags more.
- Equalize **perturbation RMS** across attacks, or the broadband one wins for free.
- Pick thresholds on a **dev image** and report on held-out ones; per-image oracle thresholds
  inflated an IoU of 0.1 into an apparent 0.47.